# Production protocol and optional short DE demonstration

Document the exact completed experiment and, separately, offer a deliberately small demonstration that is **not** part of the 3000-run scientific dataset.

**Data:** the frozen 3000-run production set (`data/raw`). **Acceptance:** all five normalized residual magnitudes strictly below one. These are *numerical widths*, not independently measured Gaussian standard deviations.

> Start from the repository root or its `notebooks/` folder. No recomputation of the 3000 DE optimizations is required.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Works in Jupyter, VS Code, and Colab if the repository is cloned locally.
workdir = Path.cwd().resolve()
candidates = [workdir, workdir.parent]
ROOT = next((p for p in candidates if (p / "data/raw/runs.csv").is_file()
             and (p / "src/model.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError(
        "Cannot find the repository root. Open the project folder (with data/ and src/) "
        "or clone the repository in the Colab session before running the notebook."
    )
sys.path.insert(0, str(ROOT / "src"))
from model import PARAMETERS, V_EW, MASS_WIDTHS, mass_matrix, spectrum
raw = pd.read_csv(ROOT / "data/raw/runs.csv")
print(f"Repository: {ROOT.name}; records: {len(raw):,}; v = {V_EW} GeV")

Repository: DE_Particle_paper_public; records: 3,000; v = 246.22 GeV


In [2]:
import json
config=json.loads((ROOT/'data/raw/production_config.json').read_text())
print(json.dumps(config,indent=2))

{
  "preset": "production",
  "n_targets": 1000,
  "n_restarts": 3,
  "restart_seeds": [
    11,
    23,
    37
  ],
  "target_seed": 20261020,
  "maxiter": 300,
  "popsize": 50,
  "tol": 0.01,
  "polish": true,
  "v_ew": 246.22,
  "mass_centers": [
    95.4,
    125.09,
    151.5
  ],
  "mass_widths": [
    0.05,
    0.09,
    0.09
  ],
  "optimizer_charged_mass_squared_residual": [
    -260,
    260
  ],
  "optimizer_kappa_range": [
    0.98,
    1.08
  ],
  "analysis_z_threshold": 2,
  "python": "3.12.1",
  "scipy": "1.11.4",
  "numpy": "1.26.2",
  "version_note": "Fresh numerical experiment; never merge with legacy results."
}


## Important: internal constraints differ from the five final cuts\nThe original optimizer constrained \(|M_{H^\pm}^2-130^2|\le 260\,\mathrm{GeV}^2\) and \(0.98\le\kappa_{2V}\le1.08\). The publication sample is subsequently selected using five strict \(z<1\) criteria, where \(\sigma=(0.05,0.09,0.09)\,\mathrm{GeV}\) for neutral masses, \(\sigma_\pm=1\,\mathrm{GeV}\), and \(\sigma_\kappa=0.05\). The 3000 optimizations were not rerun after changing the final post-processing selection.

In [3]:
from scipy.optimize import differential_evolution, NonlinearConstraint
from model import spectral_loss, constraints, BOUNDS
from time import perf_counter

target=raw.loc[0,["target_H1","target_H2","target_H3"]].to_numpy(float)
print("First target (GeV):",target)
print("Original frozen run count:",len(raw))
print("For a full production run, see src/run_de.py (1000 targets × 3 restarts).")

First target (GeV): [ 95.30418795 125.07836228 151.310701  ]
Original frozen run count: 3000
For a full production run, see src/run_de.py (1000 targets × 3 restarts).


## Optional tiny DE example (disabled by default)\nThis quick example uses a tiny generation budget and is **not** a substitute for the archived 3000 runs. Change `RUN_DEMO=True` to execute; its output is not added to the data files. It may or may not reach the final accuracy threshold.

In [4]:
RUN_DEMO = False
if RUN_DEMO:
    condition=NonlinearConstraint(constraints,[-260,.98],[260,1.08])
    result=differential_evolution(spectral_loss,BOUNDS,args=(target,),constraints=(condition,),
        maxiter=5,popsize=5,polish=False,seed=11)
    print("Illustrative short-run objective:",result.fun)
    print("Short-run predicted neutral masses (GeV):",spectrum(result.x)[0])
else:
    print("DE demo skipped (set RUN_DEMO=True to opt in).")

DE demo skipped (set RUN_DEMO=True to opt in).


## Reproduce the archived data audit from the command line\nFrom the root folder execute `python scripts/verify_dataset.py`. For a *new* expensive scan, `python src/run_de.py --preset production --targets 1000 --restarts 3 --output data/new_runs/new_study` uses a fresh output folder. It does not alter the archived CSVs.